# 01：数据集与 RAG 数据结构

## 本节目标

本节只用少量人工数据理解 RAG 评测所需的最小结构，不下载完整数据集，也不调用模型。完成后，你应该能区分普通问答样本与可用于检索评测的 RAG 样本。

## 四个最小概念

- **Document（文档）**：系统可以搜索的原始知识单元，例如一段产品手册。
- **Query（查询）**：用户提出、需要检索系统理解的问题。
- **Answer（答案）**：期望回答或参考答案，用于检查最终回答质量。
- **Relevant Document（相关文档）**：已知能支持该查询答案的文档，是计算 Recall@k、MRR 等检索指标的依据。

一个完整的可评测 RAG 样本不仅要有问题和答案，还要知道哪些文档应该被检索到。

In [ ]:
# 输入：当前 Python 环境。输出：解释器路径和核心依赖版本，便于复现实验。
import sys
import datasets
import matplotlib
import numpy as np
import pandas as pd
import sklearn

versions = {
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "datasets": datasets.__version__,
}
versions

## 创建人工模拟数据

下面先建立 4 条文档，再建立查询、参考答案和相关文档标注。输入是人工编写的 Python 列表；输出是两个表格。

In [ ]:
documents = [
    {"document_id": "doc-001", "title": "账户安全", "text": "连续五次输入错误密码后，账户会锁定十分钟。"},
    {"document_id": "doc-002", "title": "密码重置", "text": "用户可在登录页通过已验证邮箱重置密码。"},
    {"document_id": "doc-003", "title": "订单取消", "text": "未发货订单可以在订单详情页直接取消。"},
    {"document_id": "doc-004", "title": "退款时间", "text": "退款审核通过后，款项通常在三到五个工作日内原路退回。"},
]

rag_examples = [
    {"query_id": "q-001", "query": "密码输错太多次会怎样？", "answer": "连续输错五次后账户锁定十分钟。", "relevant_document_ids": ["doc-001"]},
    {"query_id": "q-002", "query": "忘记密码后如何找回？", "answer": "在登录页使用已验证邮箱重置密码。", "relevant_document_ids": ["doc-002"]},
    {"query_id": "q-003", "query": "未发货的订单能取消吗？", "answer": "可以在订单详情页直接取消。", "relevant_document_ids": ["doc-003"]},
    {"query_id": "q-004", "query": "退款审核通过后多久能到账？", "answer": "通常需要三到五个工作日。", "relevant_document_ids": ["doc-004"]},
]

In [ ]:
# DataFrame 只负责把列表展示成表格，不改变原始数据。
document_table = pd.DataFrame(documents)
evaluation_table = pd.DataFrame(rag_examples)

display(document_table)
display(evaluation_table)

## 问答数据与可评测 RAG 数据的区别

| 数据类型 | 必需字段 | 能评测什么 |
|---|---|---|
| 问答数据 | Query、Answer | 最终回答与参考答案的接近程度 |
| 可评测 RAG 数据 | Document 集合、Query、Answer、Relevant Document 标注 | 检索是否找对文档，以及最终回答是否正确 |

如果缺少 Relevant Document 标注，即使最终答案错了，也很难判断错误来自检索阶段还是生成阶段。

## 小练习

为 `documents` 增加一条“修改收货地址”的文档，再为 `rag_examples` 增加一个对应问题。请确认新样本同时具有 `answer` 和 `relevant_document_ids`，然后重新运行表格单元格。

思考：如果只写答案、不写相关文档 ID，我们还能判断检索器有没有找对依据吗？

## 下一步

下一阶段会先确认 RAGBench 的当前数据说明与许可，再只接入 `emanual` 子集所需的小规模样本。我们会显式查看字段、划分和文档—查询关联，不会直接下载完整大型数据或把处理过程交给黑盒框架。